# CodeMix-T — Phase 1: Data Collection & Preprocessing

This notebook covers the full data pipeline:
1. Install dependencies
2. Download all data sources
3. Explore and understand each dataset
4. Clean and normalize text
5. Build parallel sentence pairs (code-mixed → English)
6. Language ID tagging (per token)
7. Train/Val/Test split
8. Save final datasets
9. Train custom SentencePiece BPE tokenizer
10. Validate tokenizer output

**Target:** ~50K–100K clean parallel sentence pairs ready for model training.

## Cell 1 — Install Dependencies

In [1]:
pip install torch numpy pandas tqdm sentencepiece datasets wandb sacrebleu transformers gradio fastapi pydantic uvicorn kaggle langdetect indic-nlp-library aksharamukha

  Obtaining dependency information for torch from https://files.pythonhosted.org/packages/d1/bd/9912d30b68845256aabbb4a40aeefeef3c3b20db5211ccda653544ada4b6/torch-2.11.0-cp311-cp311-win_amd64.whl.metadata
  Obtaining dependency information for sentencepiece from https://files.pythonhosted.org/packages/32/b8/f709977f5fda195ae1ea24f24e7c581163b6f142b1005bc3d0bbfe4d7082/sentencepiece-0.2.1-cp311-cp311-win_amd64.whl.metadata
  Obtaining dependency information for datasets from https://files.pythonhosted.org/packages/b0/e5/247d094108e42ac26363ab8dc57f168840cf7c05774b40ffeb0d78868fcc/datasets-4.8.4-py3-none-any.whl.metadata
  Obtaining dependency information for wandb from https://files.pythonhosted.org/packages/89/22/680d34c1587f3a979c701b66d71aa7c42b4ef2fdf0774f67034e618e834e/wandb-0.25.1-py3-none-win_amd64.whl.metadata
  Obtaining dependency information for sacrebleu from https://files.pythonhosted.org/packages/06/f2/6c90ccf3ad1d09a7d662a405b274f3c93b92df59c8d6a025d26aaf34d302/sacrebleu-2

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
conda-repo-cli 1.0.41 requires requests_mock, which is not installed.
anaconda-cloud-auth 0.1.4 requires pydantic<2.0, but you have pydantic 2.12.5 which is incompatible.
conda-repo-cli 1.0.41 requires clyent==1.2.1, but you have clyent 1.2.2 which is incompatible.
conda-repo-cli 1.0.41 requires nbformat==5.4.0, but you have nbformat 5.7.0 which is incompatible.
conda-repo-cli 1.0.41 requires requests==2.28.1, but you have requests 2.33.1 which is incompatible.
jupyter-server 1.23.4 requires anyio<4,>=3.1.0, but you have anyio 4.13.0 which is incompatible.
mysql-connector-python 8.0.33 requires protobuf<=3.20.3,>=3.11.0, but you have protobuf 6.33.6 which is incompatible.
python-lsp-black 1.2.1 requires black>=22.3.0, but you have black 0.0 which is incompatible.
s3fs 2023.3.0 requires fsspec==2023.3.0, but you ha

In [ ]:
# Install all required libraries
!pip install -q datasets sentencepiece langdetect pandas numpy tqdm kaggle
!pip install -q indic-nlp-library aksharamukha

import os, re, json, random
import pandas as pd
import numpy as np
from tqdm import tqdm
from pathlib import Path

# Create directory structure
for d in ['data/raw', 'data/processed', 'data/final', 'tokenizer']:
    Path(d).mkdir(parents=True, exist_ok=True)

print('Setup complete.')

## Cell 2 — Mount Google Drive (to persist data across sessions)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Create a persistent folder in your Drive
DRIVE_DIR = '/content/drive/MyDrive/CodeMixT'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Drive mounted. Saving final outputs to: {DRIVE_DIR}')

## Cell 3 — Download Kaggle Datasets

To use the Kaggle API:
1. Go to kaggle.com → Account → Create New Token
2. Download `kaggle.json`
3. Upload it when prompted below

In [ ]:
from google.colab import files

# Upload your kaggle.json
print('Upload your kaggle.json file:')
uploaded = files.upload()

# Set up Kaggle credentials
os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
os.rename('kaggle.json', os.path.expanduser('~/.kaggle/kaggle.json'))
os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)

# Download both Kaggle datasets
!kaggle datasets download -d pk13055/code-mixed-hindienglish-dataset -p data/raw/hinglish_kaggle --unzip
!kaggle datasets download -d younusmohamed/tanglish-and-tamil-transliterated-words-dataset -p data/raw/tanglish_kaggle --unzip

print('\nKaggle downloads complete. Files:')
!find data/raw -type f | head -30

## Cell 4 — Download L3Cube-HingCorpus (Hinglish ↔ English parallel pairs)

In [ ]:
# L3Cube-HingCorpus: ~52K Hinglish<->English sentence pairs
# Direct download from GitHub
!git clone https://github.com/l3cube-pune/hindi-nlp.git data/raw/l3cube_hing

# Check what's available
!find data/raw/l3cube_hing -type f -name '*.csv' -o -name '*.txt' | head -20

## Cell 5 — Download HuggingFace Datasets

In [ ]:
from datasets import load_dataset

# --- LinCE Benchmark (code-mixed language ID + sentiment) ---
# Has Hinglish and Spanish-English code-mixed data with token-level language tags
print('Loading LinCE sa_spaeng dataset...')
try:
    lince = load_dataset('lince', 'sa_hineng', trust_remote_code=True)
    print(f'LinCE Hinglish loaded: {lince}')
    lince.save_to_disk('data/raw/lince_hineng')
except Exception as e:
    print(f'LinCE load error: {e}')
    print('Skipping LinCE — will proceed with other sources.')

# --- Samanantar: Hindi<->English parallel corpus (use small subset) ---
print('\nLoading Samanantar hi-en (subset)...')
try:
    samanantar = load_dataset('ai4bharat/samanantar', 'hi', split='train[:30000]', trust_remote_code=True)
    print(f'Samanantar loaded: {len(samanantar)} pairs')
    samanantar.save_to_disk('data/raw/samanantar_hi')
except Exception as e:
    print(f'Samanantar load error: {e}')
    print('Skipping Samanantar.')

print('\nDone.')

## Cell 6 — Explore Each Dataset

In [ ]:
# ---- Explore Kaggle Hinglish dataset ----
print('=== KAGGLE HINGLISH ===')
hinglish_files = list(Path('data/raw/hinglish_kaggle').rglob('*.*'))
print(f'Files: {hinglish_files}')

for f in hinglish_files[:3]:
    try:
        df = pd.read_csv(f, nrows=5)
        print(f'\n{f.name} — shape: {df.shape}')
        print(df.head())
        print('Columns:', df.columns.tolist())
    except:
        pass

print('\n=== KAGGLE TANGLISH ===')
tanglish_files = list(Path('data/raw/tanglish_kaggle').rglob('*.*'))
print(f'Files: {tanglish_files}')

for f in tanglish_files[:3]:
    try:
        df = pd.read_csv(f, nrows=5)
        print(f'\n{f.name} — shape: {df.shape}')
        print(df.head())
        print('Columns:', df.columns.tolist())
    except:
        pass

## Cell 7 — Data Cleaning Functions

In [ ]:
import unicodedata

def clean_text(text: str) -> str:
    """Clean a single text string."""
    if not isinstance(text, str):
        return ''

    # Normalize unicode
    text = unicodedata.normalize('NFC', text)

    # Remove URLs
    text = re.sub(r'http\S+|www\.\S+', '', text)

    # Remove HTML tags
    text = re.sub(r'<[^>]+>', '', text)

    # Remove @mentions and #hashtags content (keep the word, drop symbol)
    text = re.sub(r'[@#]', '', text)

    # Remove excessive punctuation (3+ repeated)
    text = re.sub(r'([^\w\s])\1{2,}', r'\1', text)

    # Normalize whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    return text


def is_valid_pair(src: str, tgt: str,
                  min_len: int = 3,
                  max_len: int = 128,
                  max_ratio: float = 3.0) -> bool:
    """Filter out bad sentence pairs."""
    if not src or not tgt:
        return False

    src_words = src.split()
    tgt_words = tgt.split()

    # Length filters
    if len(src_words) < min_len or len(tgt_words) < min_len:
        return False
    if len(src_words) > max_len or len(tgt_words) > max_len:
        return False

    # Length ratio filter (avoid very unbalanced pairs)
    ratio = max(len(src_words), len(tgt_words)) / min(len(src_words), len(tgt_words))
    if ratio > max_ratio:
        return False

    # Duplicate filter
    if src.strip().lower() == tgt.strip().lower():
        return False

    return True


def has_code_mixing(text: str) -> bool:
    """
    Basic check: does text contain both Latin script and
    Devanagari / Tamil characters, OR is it Romanized (transliterated)
    code-mixed (all Latin but with Hindi/Tamil words)?
    We keep all-Latin text too since Romanized code-mixing is common.
    """
    has_devanagari = bool(re.search(r'[\u0900-\u097F]', text))
    has_tamil      = bool(re.search(r'[\u0B80-\u0BFF]', text))
    has_latin      = bool(re.search(r'[a-zA-Z]', text))

    # Script-mixed (Devanagari/Tamil + Latin)
    if (has_devanagari or has_tamil) and has_latin:
        return True

    # Romanized code-mixed: all Latin — keep these too
    if has_latin:
        return True

    return False


print('Cleaning functions defined.')

# Quick sanity tests
samples = [
    ('kal main market gaya tha', 'I went to the market yesterday'),
    ('naan romba tired aa irukken', 'I am very tired'),
    ('', 'empty source'),
    ('hi', 'hello'),
]

for src, tgt in samples:
    src_c = clean_text(src)
    tgt_c = clean_text(tgt)
    valid = is_valid_pair(src_c, tgt_c)
    cm    = has_code_mixing(src_c)
    print(f'  src={repr(src_c):<40} valid={valid}, code_mixed={cm}')

## Cell 8 — Process Kaggle Hinglish Dataset

In [ ]:
hinglish_pairs = []

hinglish_files = list(Path('data/raw/hinglish_kaggle').rglob('*.csv'))
print(f'Found {len(hinglish_files)} CSV files in Hinglish Kaggle dataset')

for f in hinglish_files:
    try:
        df = pd.read_csv(f)
        print(f'\nProcessing {f.name} — columns: {df.columns.tolist()}')

        # Auto-detect source and target columns
        # Common column names to look for:
        src_candidates = ['hinglish', 'hindi_english', 'code_mixed', 'source', 'text', 'sentence']
        tgt_candidates = ['english', 'translation', 'target', 'en', 'eng']

        src_col = next((c for c in df.columns if c.lower() in src_candidates), None)
        tgt_col = next((c for c in df.columns if c.lower() in tgt_candidates), None)

        # If no exact match, show columns for manual inspection
        if not src_col or not tgt_col:
            print(f'  Could not auto-detect columns. Available: {df.columns.tolist()}')
            print('  Preview:')
            print(df.head(3))
            # Manually set here if needed:
            # src_col = 'YOUR_COLUMN'
            # tgt_col = 'YOUR_COLUMN'
            continue

        print(f'  Using: src={src_col}, tgt={tgt_col}')

        for _, row in tqdm(df.iterrows(), total=len(df), desc=f.name):
            src = clean_text(str(row[src_col]))
            tgt = clean_text(str(row[tgt_col]))
            if is_valid_pair(src, tgt):
                hinglish_pairs.append({
                    'source': src,
                    'target': tgt,
                    'language': 'hinglish',
                    'split': 'train'
                })
    except Exception as e:
        print(f'  Error processing {f.name}: {e}')

print(f'\nTotal Hinglish pairs collected: {len(hinglish_pairs)}')
if hinglish_pairs:
    print('Sample:')
    for p in hinglish_pairs[:3]:
        print(f'  SRC: {p["source"]}')
        print(f'  TGT: {p["target"]}')
        print()

## Cell 9 — Process Kaggle Tanglish Dataset

In [ ]:
tanglish_vocab = []  # word-level transliteration pairs for tokenizer
tanglish_pairs = []  # sentence pairs if any exist

tanglish_files = list(Path('data/raw/tanglish_kaggle').rglob('*.csv'))
print(f'Found {len(tanglish_files)} CSV files in Tanglish Kaggle dataset')

for f in tanglish_files:
    try:
        df = pd.read_csv(f)
        print(f'\nProcessing {f.name} — shape: {df.shape}')
        print(f'  Columns: {df.columns.tolist()}')
        print(df.head(3))

        # This dataset is word-level transliteration
        # Collect all text for tokenizer training vocabulary
        for col in df.columns:
            texts = df[col].dropna().astype(str).tolist()
            tanglish_vocab.extend([clean_text(t) for t in texts if clean_text(t)])

    except Exception as e:
        print(f'  Error: {e}')

print(f'\nTanglish vocabulary tokens collected: {len(tanglish_vocab)}')
print('Sample:', tanglish_vocab[:10])

## Cell 10 — Process L3Cube HingCorpus

In [ ]:
l3cube_pairs = []

# L3Cube repo structure — look for translation files
l3cube_files = list(Path('data/raw/l3cube_hing').rglob('*.csv'))
print(f'L3Cube files found: {l3cube_files}')

for f in l3cube_files:
    try:
        df = pd.read_csv(f)
        print(f'\n{f.name} — shape: {df.shape}, columns: {df.columns.tolist()}')
        print(df.head(3))

        # L3Cube typically has 'Hinglish' and 'English' columns
        src_col = next((c for c in df.columns if 'hing' in c.lower() or 'hindi' in c.lower() or 'source' in c.lower()), None)
        tgt_col = next((c for c in df.columns if 'eng' in c.lower() or 'target' in c.lower()), None)

        if src_col and tgt_col:
            print(f'  Using: src={src_col}, tgt={tgt_col}')
            for _, row in tqdm(df.iterrows(), total=len(df), desc='L3Cube'):
                src = clean_text(str(row[src_col]))
                tgt = clean_text(str(row[tgt_col]))
                if is_valid_pair(src, tgt):
                    l3cube_pairs.append({
                        'source': src,
                        'target': tgt,
                        'language': 'hinglish',
                        'split': 'train'
                    })
        else:
            print(f'  Could not detect columns. Manual inspection needed.')
    except Exception as e:
        print(f'  Error: {e}')

print(f'\nL3Cube pairs collected: {len(l3cube_pairs)}')

## Cell 11 — Process Samanantar (Hindi ↔ English)

In [ ]:
from datasets import load_from_disk

samanantar_pairs = []

try:
    ds = load_from_disk('data/raw/samanantar_hi')
    print(f'Samanantar loaded: {len(ds)} pairs')
    print('Columns:', ds.column_names)
    print('Sample:', ds[0])

    # Samanantar columns: 'src' (Hindi) and 'tgt' (English)
    for item in tqdm(ds, desc='Samanantar'):
        src = clean_text(item.get('src', '') or item.get('sentence1', ''))
        tgt = clean_text(item.get('tgt', '') or item.get('sentence2', ''))
        if is_valid_pair(src, tgt):
            samanantar_pairs.append({
                'source': src,
                'target': tgt,
                'language': 'hindi',  # pure Hindi, not code-mixed — used for augmentation
                'split': 'train'
            })

    print(f'\nSamanantar pairs collected: {len(samanantar_pairs)}')
except Exception as e:
    print(f'Could not load Samanantar: {e}')
    print('Skipping — continuing without it.')

## Cell 12 — Merge All Datasets & Deduplicate

In [ ]:
all_pairs = hinglish_pairs + l3cube_pairs + samanantar_pairs + tanglish_pairs

print(f'Total before deduplication: {len(all_pairs)}')
print(f'  Hinglish (Kaggle): {len(hinglish_pairs)}')
print(f'  L3Cube:            {len(l3cube_pairs)}')
print(f'  Samanantar:        {len(samanantar_pairs)}')
print(f'  Tanglish:          {len(tanglish_pairs)}')

# Convert to DataFrame
df_all = pd.DataFrame(all_pairs)

# Deduplicate on source sentence
before = len(df_all)
df_all = df_all.drop_duplicates(subset=['source'])
after = len(df_all)
print(f'\nAfter deduplication: {after} (removed {before - after} duplicates)')

# Shuffle
df_all = df_all.sample(frac=1, random_state=42).reset_index(drop=True)

print('\nLanguage distribution:')
print(df_all['language'].value_counts())

print('\nSample pairs:')
print(df_all[['source', 'target', 'language']].head(10).to_string())

## Cell 13 — Token-Level Language ID Tagging

This is the key preprocessing step for our **Language-ID-Aware Embeddings**.
Each token gets a tag: `LANG_EN`, `LANG_HI`, or `LANG_TA`.

In [ ]:
import unicodedata

# Language ID tag constants
LANG_EN = 0
LANG_HI = 1
LANG_TA = 2
LANG_UNK = 3

LANG_NAMES = {0: 'EN', 1: 'HI', 2: 'TA', 3: 'UNK'}

# Common Hinglish words (Romanized Hindi that appear in Latin script)
HINDI_ROMANIZED_KEYWORDS = {
    'hai', 'hain', 'tha', 'thi', 'the', 'main', 'mein', 'nahi', 'nahin',
    'kya', 'koi', 'aur', 'lekin', 'par', 'toh', 'bhi', 'abhi', 'yahan',
    'wahan', 'kab', 'kyun', 'kaisa', 'kaisi', 'bahut', 'accha', 'theek',
    'kal', 'aaj', 'subah', 'raat', 'gaya', 'aya', 'dekh', 'baat', 'karo',
    'hoga', 'hogi', 'karke', 'leke', 'jaake', 'rehna', 'matlab', 'matlab',
    'samajh', 'pata', 'paise', 'kaam', 'log', 'dost', 'yaar', 'bhai'
}

# Common Tanglish words (Romanized Tamil)
TAMIL_ROMANIZED_KEYWORDS = {
    'naan', 'nee', 'avan', 'aval', 'avanga', 'romba', 'sollu', 'solla',
    'paar', 'paaru', 'vaa', 'po', 'enna', 'epdi', 'eppdi', 'eppova',
    'konjam', 'koncham', 'theriyum', 'theriyala', 'irukku', 'irukken',
    'pannuven', 'pannrom', 'seri', 'illa', 'illai', 'ama', 'aama',
    'enna', 'yenna', 'inge', 'anga', 'yaar', 'yaaru', 'enga', 'venum',
    'vendam', 'mudiyadhu', 'paavam', 'sappa', 'kalakku', 'super'
}


def get_token_lang_id(token: str, sentence_lang: str = 'hinglish') -> int:
    """
    Assign a language ID to a single token.
    Uses Unicode block detection for script-mixed text,
    and keyword lookup for Romanized code-mixed text.
    """
    token_lower = token.lower().strip()

    # Check Unicode blocks
    has_devanagari = any('\u0900' <= c <= '\u097F' for c in token)
    has_tamil_script = any('\u0B80' <= c <= '\u0BFF' for c in token)
    has_latin = any(c.isascii() and c.isalpha() for c in token)

    if has_devanagari:
        return LANG_HI
    if has_tamil_script:
        return LANG_TA

    # Romanized token — use keyword lookup
    if has_latin:
        if token_lower in HINDI_ROMANIZED_KEYWORDS:
            return LANG_HI
        if token_lower in TAMIL_ROMANIZED_KEYWORDS:
            return LANG_TA
        # Default Romanized to English
        return LANG_EN

    return LANG_UNK


def tag_sentence(sentence: str, lang: str = 'hinglish') -> list:
    """
    Tokenize a sentence and return list of (token, lang_id) tuples.
    """
    tokens = sentence.split()
    return [(tok, get_token_lang_id(tok, lang)) for tok in tokens]


# Test it
test_sentences = [
    ('kal main market gaya tha for shopping', 'hinglish'),
    ('naan romba tired aa irukken today', 'tanglish'),
    ('yaar bahut accha movie tha', 'hinglish'),
    ('konjam wait panna sollu', 'tanglish'),
]

print('Token-level Language ID tagging demo:\n')
for sent, lang in test_sentences:
    tagged = tag_sentence(sent, lang)
    print(f'Input ({lang}): {sent}')
    print('Tagged: ', ' '.join([f'{tok}[{LANG_NAMES[lid]}]' for tok, lid in tagged]))
    print()

## Cell 14 — Apply Language ID Tags to Full Dataset

In [ ]:
def apply_lang_tags(row):
    tagged = tag_sentence(row['source'], row['language'])
    tokens = [t for t, _ in tagged]
    lang_ids = [l for _, l in tagged]
    return pd.Series({
        'source_tokens': ' '.join(tokens),
        'source_lang_ids': ' '.join(map(str, lang_ids))
    })

print('Applying language ID tags to all pairs...')
tqdm.pandas(desc='Tagging')
lang_tag_cols = df_all.progress_apply(apply_lang_tags, axis=1)
df_all = pd.concat([df_all, lang_tag_cols], axis=1)

print('Done. Sample:')
print(df_all[['source', 'source_lang_ids', 'target']].head(5).to_string())

## Cell 15 — Train / Validation / Test Split

In [ ]:
total = len(df_all)

# 90 / 5 / 5 split
train_end = int(total * 0.90)
val_end   = int(total * 0.95)

df_train = df_all.iloc[:train_end].copy()
df_val   = df_all.iloc[train_end:val_end].copy()
df_test  = df_all.iloc[val_end:].copy()

df_train['split'] = 'train'
df_val['split']   = 'val'
df_test['split']  = 'test'

print(f'Split complete:')
print(f'  Train: {len(df_train)}')
print(f'  Val:   {len(df_val)}')
print(f'  Test:  {len(df_test)}')

# Save to processed
df_train.to_csv('data/processed/train.csv', index=False)
df_val.to_csv('data/processed/val.csv', index=False)
df_test.to_csv('data/processed/test.csv', index=False)

print('\nSaved to data/processed/')

## Cell 16 — Prepare Tokenizer Training Corpus

In [ ]:
# Write all source + target text into a flat file for tokenizer training
tokenizer_corpus_path = 'data/processed/tokenizer_corpus.txt'

with open(tokenizer_corpus_path, 'w', encoding='utf-8') as f:
    # Add all source sentences (code-mixed)
    for text in df_train['source'].dropna():
        f.write(text.strip() + '\n')

    # Add all target sentences (English)
    for text in df_train['target'].dropna():
        f.write(text.strip() + '\n')

    # Add Tanglish vocabulary words (boosts Tamil-script coverage)
    for text in tanglish_vocab:
        if text.strip():
            f.write(text.strip() + '\n')

# Count lines
with open(tokenizer_corpus_path) as f:
    n_lines = sum(1 for _ in f)

print(f'Tokenizer corpus written: {n_lines:,} lines')
print(f'File: {tokenizer_corpus_path}')

## Cell 17 — Train Custom SentencePiece BPE Tokenizer

In [ ]:
import sentencepiece as spm

# Special tokens needed for the model
SPECIAL_TOKENS = [
    '<pad>',    # 0 — padding
    '<unk>',    # 1 — unknown
    '<bos>',    # 2 — beginning of sequence
    '<eos>',    # 3 — end of sequence
    '<lang_en>',# 4 — English language tag
    '<lang_hi>',# 5 — Hindi language tag
    '<lang_ta>',# 6 — Tamil language tag
    '<translate>', # 7 — chatbot instruction token
]

# Train BPE tokenizer
print('Training SentencePiece BPE tokenizer...')
spm.SentencePieceTrainer.train(
    input=tokenizer_corpus_path,
    model_prefix='tokenizer/codemix_bpe',
    vocab_size=16000,          # 16K vocab — good for code-mixed + English
    model_type='bpe',
    character_coverage=0.9999, # Cover rare Unicode chars (Tamil, Devanagari)
    pad_id=0,
    unk_id=1,
    bos_id=2,
    eos_id=3,
    user_defined_symbols=SPECIAL_TOKENS[4:],  # pad/unk/bos/eos handled above
    num_threads=4,
    input_sentence_size=500000,  # Cap to avoid OOM
    shuffle_input_sentence=True,
)

print('Tokenizer training complete!')
print('Files saved: tokenizer/codemix_bpe.model, tokenizer/codemix_bpe.vocab')

## Cell 18 — Validate Tokenizer

In [ ]:
# Load trained tokenizer
sp = spm.SentencePieceProcessor()
sp.load('tokenizer/codemix_bpe.model')

print(f'Vocabulary size: {sp.get_piece_size()}')
print(f'PAD id: {sp.pad_id()}')
print(f'BOS id: {sp.bos_id()}')
print(f'EOS id: {sp.eos_id()}')
print(f'UNK id: {sp.unk_id()}')
print()

# Test on various code-mixed sentences
test_cases = [
    'kal main market gaya tha for shopping',
    'naan romba tired aa irukken today',
    'yaar bahut accha movie tha yesterday',
    'konjam wait panna sollu please',
    'I went to the market yesterday',
]

print('Tokenization examples:\n')
for text in test_cases:
    tokens = sp.encode_as_pieces(text)
    ids    = sp.encode_as_ids(text)
    decoded = sp.decode_pieces(tokens)
    print(f'Input:   {text}')
    print(f'Tokens:  {tokens}')
    print(f'IDs:     {ids}')
    print(f'Decoded: {decoded}')
    print(f'OOV?     {any(i == sp.unk_id() for i in ids)}')
    print()

## Cell 19 — Tokenize Full Dataset & Save Final Files

In [ ]:
def tokenize_pair(row):
    """Tokenize a source-target pair and return token ID sequences."""
    src_ids = [sp.bos_id()] + sp.encode_as_ids(row['source']) + [sp.eos_id()]
    tgt_ids = [sp.bos_id()] + sp.encode_as_ids(row['target']) + [sp.eos_id()]
    return pd.Series({
        'src_ids': src_ids,
        'tgt_ids': tgt_ids,
        'src_len': len(src_ids),
        'tgt_len': len(tgt_ids)
    })

print('Tokenizing train set...')
train_tok = df_train.progress_apply(tokenize_pair, axis=1)
df_train_final = pd.concat([df_train, train_tok], axis=1)

print('Tokenizing val set...')
val_tok = df_val.progress_apply(tokenize_pair, axis=1)
df_val_final = pd.concat([df_val, val_tok], axis=1)

print('Tokenizing test set...')
test_tok = df_test.progress_apply(tokenize_pair, axis=1)
df_test_final = pd.concat([df_test, test_tok], axis=1)

# Save final datasets
df_train_final.to_json('data/final/train.jsonl', orient='records', lines=True)
df_val_final.to_json('data/final/val.jsonl', orient='records', lines=True)
df_test_final.to_json('data/final/test.jsonl', orient='records', lines=True)

print('\nFinal datasets saved to data/final/')
print(f'  train.jsonl: {len(df_train_final)} samples')
print(f'  val.jsonl:   {len(df_val_final)} samples')
print(f'  test.jsonl:  {len(df_test_final)} samples')

# Sequence length stats
print('\nSequence length stats (src):')
print(df_train_final['src_len'].describe())
print('\nSequence length stats (tgt):')
print(df_train_final['tgt_len'].describe())

## Cell 20 — Copy Everything to Google Drive

In [ ]:
import shutil

# Copy final data and tokenizer to Drive
shutil.copytree('data/final', f'{DRIVE_DIR}/data_final', dirs_exist_ok=True)
shutil.copytree('tokenizer', f'{DRIVE_DIR}/tokenizer', dirs_exist_ok=True)

print('Saved to Google Drive:')
print(f'  {DRIVE_DIR}/data_final/')
print(f'  {DRIVE_DIR}/tokenizer/')

print('\n=== PHASE 1 COMPLETE ===')
print(f'Total training pairs: {len(df_train_final)}')
print(f'Vocabulary size:      {sp.get_piece_size()}')
print(f'Tokenizer model:      tokenizer/codemix_bpe.model')
print('Ready for Phase 2: Architecture Implementation')

## Cell 21 — Dataset Summary Report

In [ ]:
print('=' * 60)
print('CODEMIX-T  |  PHASE 1 SUMMARY REPORT')
print('=' * 60)

print('\nDATA SOURCES')
print(f'  Kaggle Hinglish Dataset:  {len(hinglish_pairs):>8,} pairs')
print(f'  L3Cube HingCorpus:        {len(l3cube_pairs):>8,} pairs')
print(f'  Samanantar Hindi-EN:      {len(samanantar_pairs):>8,} pairs')
print(f'  Tanglish (vocab only):    {len(tanglish_vocab):>8,} tokens')
print(f'  Total after dedup:        {len(df_all):>8,} pairs')

print('\nSPLITS')
print(f'  Train: {len(df_train_final):>8,}')
print(f'  Val:   {len(df_val_final):>8,}')
print(f'  Test:  {len(df_test_final):>8,}')

print('\nTOKENIZER')
print(f'  Type:         SentencePiece BPE')
print(f'  Vocab size:   {sp.get_piece_size():,}')
print(f'  Special tokens: <pad> <unk> <bos> <eos> <lang_en> <lang_hi> <lang_ta> <translate>')

print('\nLANGUAGE DISTRIBUTION (train)')
print(df_train_final['language'].value_counts().to_string())

print('\nNEXT STEP: Phase 2 — Custom Transformer Architecture')
print('=' * 60)